# MIMIC-IV Data Exploration

## TrialMatch AI — Day 2

### Objective

Explore the MIMIC-IV demo dataset and identify the patient-level tables and variables that may be required for clinical trial eligibility matching.

This notebook focuses on:

- patient demographics
- hospital admissions
- diagnoses
- laboratory measurements
- procedures
- medications

The goal is to understand how patient information is represented and linked across MIMIC-IV before building any eligibility-matching logic.


In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../mimic-iv-clinical-database-demo-2.2")

patients = pd.read_csv(
    DATA_DIR / "hosp" / "patients.csv.gz"
)

admissions = pd.read_csv(
    DATA_DIR / "hosp" / "admissions.csv.gz"
)

print("Patients:", patients.shape)
print("Admissions:", admissions.shape)

Patients: (100, 6)
Admissions: (275, 16)


In [2]:
patients.shape
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10014729,F,21,2125,2011 - 2013,NaN
1,10003400,F,72,2134,2011 - 2013,2137-09-02
2,10002428,F,80,2155,2011 - 2013,NaN
3,10032725,F,38,2143,2011 - 2013,2143-03-30
4,10027445,F,48,2142,2011 - 2013,2146-02-09


In [3]:
admissions.shape
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaN,URGENT,P41R5N,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,?,MARRIED,HISPANIC/LATINO - PUERTO RICAN,NaN,NaN,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,NaN,URGENT,P233F6,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,2134-08-17 16:24:00,2134-08-18 03:15:00,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,URGENT,P38TI6,TRANSFER FROM HOSPITAL,DIED,Medicaid,ENGLISH,NaN,UNKNOWN,NaN,NaN,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaN,URGENT,P07HDB,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,NaN,NaN,0


## Part 2 — Diagnoses

- `diagnoses_icd` contains billed diagnoses associated with hospitalizations.
- `d_icd_diagnoses` translates ICD codes into readable descriptions.
- `seq_num` is an approximate ordering and should not be treated as a perfect importance ranking.
- Absence of a diagnosis code is not proof that the disease is absent.

In [4]:
diagnoses = pd.read_csv(DATA_DIR / "hosp" / "diagnoses_icd.csv.gz")
diagnosis_dictionary = pd.read_csv(DATA_DIR / "hosp" / "d_icd_diagnoses.csv.gz")

print("Diagnoses:", diagnoses.shape)
print("Diagnosis dictionary:", diagnosis_dictionary.shape)

Diagnoses: (4506, 5)
Diagnosis dictionary: (109775, 3)


In [5]:
diagnoses.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10035185,22580999,3,4139,9
1,10035185,22580999,10,V707,9
2,10035185,22580999,1,41401,9
3,10035185,22580999,9,3899,9
4,10035185,22580999,11,V8532,9


In [6]:
diagnosis_dictionary.head()

,icd_code,icd_version,long_title
0,0090,9,"Infectious colitis, enteritis, and gastroenter..."
1,01160,9,"Tuberculous pneumonia [any form], unspecified"
2,01186,9,"Other specified pulmonary tuberculosis, tuberc..."
3,01200,9,"Tuberculous pleurisy, unspecified"
4,01236,9,"Tuberculous laryngitis, tubercle bacilli not f..."


In [7]:
print("diagnoses columns:", diagnoses.columns.tolist())
print("diagnosis_dictionary columns:", diagnosis_dictionary.columns.tolist())

diagnoses columns: ['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version']
diagnosis_dictionary columns: ['icd_code', 'icd_version', 'long_title']


In [8]:
diagnoses_labeled = diagnoses.merge(
    diagnosis_dictionary,
    on=["icd_code", "icd_version"],
    how="left",
)

diagnoses_labeled[[
    "subject_id",
    "hadm_id",
    "seq_num",
    "icd_code",
    "icd_version",
    "long_title",
]].head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10035185,22580999,3,4139,9,Other and unspecified angina pectoris
1,10035185,22580999,10,V707,9,Examination of participant in clinical trial
2,10035185,22580999,1,41401,9,Coronary atherosclerosis of native coronary ar...
3,10035185,22580999,9,3899,9,Unspecified hearing loss
4,10035185,22580999,11,V8532,9,"Body Mass Index 32.0-32.9, adult"


In [9]:
dx_subject = diagnoses["subject_id"].iloc[0]

diagnoses_labeled.loc[
    diagnoses_labeled["subject_id"] == dx_subject,
    ["hadm_id", "seq_num", "icd_code", "icd_version", "long_title"],
].sort_values(["hadm_id", "seq_num"])

,hadm_id,seq_num,icd_code,icd_version,long_title
2,22580999,1,41401,9,Coronary atherosclerosis of native coronary ar...
5,22580999,2,25002,9,Diabetes mellitus without mention of complicat...
0,22580999,3,4139,9,Other and unspecified angina pectoris
8,22580999,4,2724,9,Other and unspecified hyperlipidemia
10,22580999,5,27800,9,"Obesity, unspecified"
9,22580999,6,V5867,9,Long-term (current) use of insulin
6,22580999,7,71696,9,"Arthropathy, unspecified, lower leg"
7,22580999,8,V1083,9,Personal history of other malignant neoplasm o...
3,22580999,9,3899,9,Unspecified hearing loss
1,22580999,10,V707,9,Examination of participant in clinical trial


## Part 3 — Laboratory measurements

- `itemid` identifies the lab concept; `d_labitems` provides the readable label.
- `valuenum` is useful for numerical eligibility comparisons.
- Units and timing must always be checked.
- Missing lab evidence should not automatically be treated as normal.

In [10]:
labevents = pd.read_csv(DATA_DIR / "hosp" / "labevents.csv.gz")
lab_dictionary = pd.read_csv(DATA_DIR / "hosp" / "d_labitems.csv.gz")

print("Labevents:", labevents.shape)
print("Lab dictionary:", lab_dictionary.shape)

Labevents: (107727, 16)
Lab dictionary: (1622, 4)


In [11]:
labevents.head()

,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,172061,10014354,29600294.0,1808066,51277,NaN,2148-08-16 00:00:00,2148-08-16 01:30:00,15.4,15.40,%,10.5,15.5,NaN,ROUTINE,NaN
1,172062,10014354,29600294.0,1808066,51279,NaN,2148-08-16 00:00:00,2148-08-16 01:30:00,3.35,3.35,m/uL,4.6,6.1,abnormal,ROUTINE,NaN
2,172068,10014354,29600294.0,1808066,52172,NaN,2148-08-16 00:00:00,2148-08-16 01:30:00,49.7,49.70,fL,35.1,46.3,abnormal,ROUTINE,NaN
3,172063,10014354,29600294.0,1808066,51301,NaN,2148-08-16 00:00:00,2148-08-16 01:30:00,20.3,20.30,K/uL,4.0,10.0,abnormal,ROUTINE,NaN
4,172050,10014354,29600294.0,1808066,51249,NaN,2148-08-16 00:00:00,2148-08-16 01:30:00,31.1,31.10,g/dL,32.0,37.0,abnormal,ROUTINE,NaN


In [12]:
lab_dictionary.head()

,itemid,label,fluid,category
0,50808,Free Calcium,Blood,Blood Gas
1,50826,Tidal Volume,Blood,Blood Gas
2,50813,Lactate,Blood,Blood Gas
3,52029,% Ionized Calcium,Blood,Blood Gas
4,50801,Alveolar-arterial Gradient,Blood,Blood Gas


In [13]:
print("labevents columns:", labevents.columns.tolist())
print("lab_dictionary columns:", lab_dictionary.columns.tolist())

labevents columns: ['labevent_id', 'subject_id', 'hadm_id', 'specimen_id', 'itemid', 'order_provider_id', 'charttime', 'storetime', 'value', 'valuenum', 'valueuom', 'ref_range_lower', 'ref_range_upper', 'flag', 'priority', 'comments']
lab_dictionary columns: ['itemid', 'label', 'fluid', 'category']


In [14]:
labs_labeled = labevents.merge(
    lab_dictionary,
    on="itemid",
    how="left",
)

lab_cols = [
    "subject_id",
    "hadm_id",
    "itemid",
    "label",
    "charttime",
    "value",
    "valuenum",
    "valueuom",
]
labs_labeled[lab_cols].head()

,subject_id,hadm_id,itemid,label,charttime,value,valuenum,valueuom
0,10014354,29600294.0,51277,RDW,2148-08-16 00:00:00,15.4,15.40,%
1,10014354,29600294.0,51279,Red Blood Cells,2148-08-16 00:00:00,3.35,3.35,m/uL
2,10014354,29600294.0,52172,RDW-SD,2148-08-16 00:00:00,49.7,49.70,fL
3,10014354,29600294.0,51301,White Blood Cells,2148-08-16 00:00:00,20.3,20.30,K/uL
4,10014354,29600294.0,51249,MCHC,2148-08-16 00:00:00,31.1,31.10,g/dL


In [15]:
lab_subject = labevents["subject_id"].iloc[0]

(
    labs_labeled.loc[
        labs_labeled["subject_id"] == lab_subject,
        ["hadm_id", "charttime", "itemid", "label", "value", "valuenum", "valueuom"],
    ]
    .sort_values("charttime")
    .head(15)
)

,hadm_id,charttime,itemid,label,value,valuenum,valueuom
77269,NaN,2146-07-17 11:40:00,50983,Sodium,138,138.0,mEq/L
77244,NaN,2146-07-17 11:40:00,51301,White Blood Cells,50.7,50.7,K/uL
77270,NaN,2146-07-17 11:40:00,51007,Uric Acid,5.0,5.0,mg/dL
77271,NaN,2146-07-17 11:40:00,50902,Chloride,102,102.0,mEq/L
77272,NaN,2146-07-17 11:40:00,50912,Creatinine,1.2,1.2,mg/dL
77273,NaN,2146-07-17 11:40:00,50976,"Protein, Total",6.3,6.3,g/dL
77274,NaN,2146-07-17 11:40:00,51006,Urea Nitrogen,20,20.0,mg/dL
77275,NaN,2146-07-17 11:40:00,50930,Globulin,2.0,2.0,g/dL
77276,NaN,2146-07-17 11:40:00,50882,Bicarbonate,27,27.0,mEq/L
77277,NaN,2146-07-17 11:40:00,50863,Alkaline Phosphatase,37,37.0,IU/L


## Part 4 — Procedures

- `procedures_icd` contains billed procedures; `d_icd_procedures` provides descriptions.
- Procedure history can support inclusion/exclusion criteria.
- Absence of a procedure code is not necessarily proof the procedure never occurred.

In [16]:
procedures = pd.read_csv(DATA_DIR / "hosp" / "procedures_icd.csv.gz")
procedure_dictionary = pd.read_csv(DATA_DIR / "hosp" / "d_icd_procedures.csv.gz")

print("Procedures:", procedures.shape)
print("Procedure dictionary:", procedure_dictionary.shape)

Procedures: (722, 6)
Procedure dictionary: (85257, 3)


In [17]:
procedures.head()

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version
0,10011398,27505812,3,2146-12-15,3961,9
1,10011398,27505812,2,2146-12-15,3615,9
2,10011398,27505812,1,2146-12-15,3614,9
3,10014729,23300884,4,2125-03-23,3897,9
4,10014729,23300884,1,2125-03-20,3403,9


In [18]:
procedure_dictionary.head()

,icd_code,icd_version,long_title
0,0039,9,Other computer assisted surgery
1,0048,9,Insertion of four or more vascular stents
2,0074,9,"Hip bearing surface, metal-on-polyethylene"
3,0077,9,"Hip bearing surface, ceramic-on-polyethylene"
4,0126,9,Insertion of catheter(s) into cranial cavity o...


In [19]:
print("procedures columns:", procedures.columns.tolist())
print("procedure_dictionary columns:", procedure_dictionary.columns.tolist())

procedures columns: ['subject_id', 'hadm_id', 'seq_num', 'chartdate', 'icd_code', 'icd_version']
procedure_dictionary columns: ['icd_code', 'icd_version', 'long_title']


In [20]:
procedures_labeled = procedures.merge(
    procedure_dictionary,
    on=["icd_code", "icd_version"],
    how="left",
)

procedures_labeled[[
    "subject_id",
    "hadm_id",
    "seq_num",
    "icd_code",
    "icd_version",
    "long_title",
]].head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10011398,27505812,3,3961,9,Extracorporeal circulation auxiliary to open h...
1,10011398,27505812,2,3615,9,Single internal mammary-coronary artery bypass
2,10011398,27505812,1,3614,9,(Aorto)coronary bypass of four or more coronar...
3,10014729,23300884,4,3897,9,Central venous catheter placement with guidance
4,10014729,23300884,1,3403,9,Reopening of recent thoracotomy site


In [21]:
proc_subject = procedures["subject_id"].iloc[0]

procedures_labeled.loc[
    procedures_labeled["subject_id"] == proc_subject,
    ["hadm_id", "seq_num", "icd_code", "icd_version", "long_title"],
].sort_values(["hadm_id", "seq_num"])

,hadm_id,seq_num,icd_code,icd_version,long_title
2,27505812,1,3614,9,(Aorto)coronary bypass of four or more coronar...
1,27505812,2,3615,9,Single internal mammary-coronary artery bypass
0,27505812,3,3961,9,Extracorporeal circulation auxiliary to open h...


## Part 5 — Medication orders

- `prescriptions` records medication orders.
- An order is not the same as confirmed administration.
- Medication timing matters.
- Missing medication records should often result in UNKNOWN rather than assuming no exposure.

In [22]:
prescriptions = pd.read_csv(DATA_DIR / "hosp" / "prescriptions.csv.gz")

print("Prescriptions:", prescriptions.shape)
print("prescriptions columns:", prescriptions.columns.tolist())

Prescriptions: (18087, 21)
prescriptions columns: ['subject_id', 'hadm_id', 'pharmacy_id', 'poe_id', 'poe_seq', 'order_provider_id', 'starttime', 'stoptime', 'drug_type', 'drug', 'formulary_drug_cd', 'gsn', 'ndc', 'prod_strength', 'form_rx', 'dose_val_rx', 'dose_unit_rx', 'form_val_disp', 'form_unit_disp', 'doses_per_24_hrs', 'route']


In [23]:
prescriptions.head()

,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,...,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
0,10027602,28166872,27168639,NaN,NaN,NaN,2201-10-30 12:00:00,NaN,MAIN,Fentanyl Citrate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10027602,28166872,40720238,NaN,NaN,NaN,2201-10-30 12:00:00,NaN,MAIN,Fentanyl Citrate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10027602,28166872,62845687,NaN,NaN,NaN,2201-10-31 12:00:00,NaN,MAIN,Lorazepam,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10027602,28166872,24340150,NaN,NaN,NaN,2201-10-30 12:00:00,NaN,MAIN,Midazolam,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10027602,28166872,14435820,NaN,NaN,NaN,2201-10-30 12:00:00,NaN,MAIN,Midazolam,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
rx_subject = prescriptions["subject_id"].iloc[0]

rx_cols = [c for c in [
    "subject_id",
    "hadm_id",
    "drug",
    "starttime",
    "stoptime",
    "route",
    "dose_val_rx",
    "dose_unit_rx",
] if c in prescriptions.columns]

(
    prescriptions.loc[prescriptions["subject_id"] == rx_subject, rx_cols]
    .sort_values("starttime")
    .head(10)
)

,subject_id,hadm_id,drug,starttime,stoptime,route,dose_val_rx,dose_unit_rx
0,10027602,28166872,Fentanyl Citrate,2201-10-30 12:00:00,NaN,NaN,NaN,NaN
1,10027602,28166872,Fentanyl Citrate,2201-10-30 12:00:00,NaN,NaN,NaN,NaN
3,10027602,28166872,Midazolam,2201-10-30 12:00:00,NaN,NaN,NaN,NaN
4,10027602,28166872,Midazolam,2201-10-30 12:00:00,NaN,NaN,NaN,NaN
11964,10027602,28166872,LeVETiracetam,2201-10-30 12:00:00,2201-10-31 14:00:00,IV,500,mg
14556,10027602,28166872,0.9% Sodium Chloride,2201-10-30 12:00:00,2201-10-31 14:00:00,IV,100,mL
9349,10027602,28166872,0.83% Sodium Chloride,2201-10-30 13:00:00,2201-10-31 14:00:00,IV DRIP,200,mL
6221,10027602,28166872,PNEUMOcoccal 23-valent polysaccharide vaccine,2201-10-30 13:00:00,2201-10-31 14:00:00,IM,0.5,mL
6102,10027602,28166872,Ondansetron,2201-10-30 13:00:00,2201-10-31 14:00:00,IV,4,mg
7453,10027602,28166872,NiCARdipine IV,2201-10-30 13:00:00,2201-10-31 14:00:00,IV DRIP,40,mg


# TrialMatch Evidence Map

The table below maps common clinical trial eligibility concepts to the MIMIC-IV tables that can supply supporting evidence for TrialMatch AI. Table roles follow the MIMIC-IV Scientific Data paper. Rows marked *(conceptual — not loaded yet)* are documented for planning only; those tables have not been loaded in this notebook.

| Eligibility concept | MIMIC table(s) | Key fields | Linking identifier(s) | What evidence this supports | Important limitation |
|---|---|---|---|---|---|
| Age / sex | `patients` | `subject_id`, `gender`, `anchor_age`, `anchor_year` | `subject_id` | Inclusion age ranges, sex-specific trials, age-based exclusions | Age is deidentified using anchor fields: `anchor_age` is the patient age at `anchor_year`, so age at any other date must be derived as `anchor_age + (year - anchor_year)` |
| Hospitalization timing | `admissions` | `subject_id`, `hadm_id`, `admittime`, `dischtime` | `subject_id`, `hadm_id` | Criteria tied to a hospital stay, e.g. events during the index admission or windows relative to admission and discharge | Admission-level rather than complete longitudinal history: care before or after the recorded stays is not visible |
| Diagnoses / comorbidities | `diagnoses_icd` + `d_icd_diagnoses` | `subject_id`, `hadm_id`, `icd_code`, `icd_version`, `long_title` | `subject_id`, `hadm_id`, plus `icd_code` + `icd_version` to the dictionary | Inclusion criteria requiring a diagnosis; exclusion criteria screening out specific comorbidities | Billing-derived diagnoses; absence of a code is not proof of absence of the condition |
| Laboratory thresholds | `labevents` + `d_labitems` | `subject_id`, `hadm_id`, `itemid`, `charttime`, `valuenum`, `valueuom`, `label` | `subject_id`, `hadm_id`, plus `itemid` to the dictionary | Numeric inclusion and exclusion thresholds, e.g. creatinine, hemoglobin, platelet or INR cutoffs | Units and timing must be checked; a lab may be missing entirely, so a threshold cannot be evaluated |
| Procedures | `procedures_icd` + `d_icd_procedures` | `subject_id`, `hadm_id`, `icd_code`, `icd_version`, `long_title` | `subject_id`, `hadm_id`, plus `icd_code` + `icd_version` to the dictionary | Prior-procedure criteria, e.g. prior revascularization required or recent surgery excluded | Billed procedure history may not represent every clinical event |
| Medication orders | `prescriptions` | `subject_id`, `hadm_id`, `drug`, `starttime`, `stoptime`, `dose`, `route` | `subject_id`, `hadm_id` | Medication-based criteria, e.g. required anticoagulant use or exclusion of a contraindicated drug | This reflects orders, not necessarily actual administration |
| Medication administration | `emar` / `emar_detail` *(conceptual — not loaded yet)* | Administration events for ordered medications, linked to pharmacy orders *(conceptual)* | `subject_id`, `hadm_id` | Confirms that an ordered medication was actually administered to the patient | Coverage depends on hospital deployment period: barcode-based administration records exist only where and when the system was in use |
| BMI / weight / blood pressure / eGFR | `omr` *(conceptual — not loaded yet)* | Intermittent ambulatory measurements such as weight, BMI, blood pressure, and calculated eGFR *(conceptual)* | `subject_id` | Anthropometric and vital-sign criteria, e.g. BMI range or blood pressure thresholds, plus derived kidney-function values | Availability varies: measurements are intermittent and absent for many patients |
| ICU-specific observations | `chartevents` + `d_items` *(conceptual — not loaded yet)* | High-resolution bedside observations during ICU stays, resolved to labels via the `d_items` dictionary *(conceptual)* | `subject_id`, `stay_id` | Physiology during ICU stays, e.g. vasopressor requirement, ventilator use, or titrated drips | Only relevant for ICU stays; patients without an ICU stay contribute no evidence from these tables |

The linking identifiers shown here are the joins the future matching engine will rely on: `subject_id` for patient-level evidence, `hadm_id` for admission-level evidence, and `stay_id` for ICU-level evidence.

# Day 2 Takeaways

- MIMIC-IV is relational and modular: separate tables cover patient identity, hospital stays, clinical events, and coded dictionaries.
- `subject_id` identifies the patient and is stable across the whole database.
- `hadm_id` identifies a hospitalization; most clinical events are anchored to it.
- Clinical trial evidence is distributed across multiple tables, so evaluating one criterion usually means joining several sources.
- Dictionary tables are needed to interpret many coded fields, e.g. ICD codes, lab itemids, and ICU itemids.
- Temporal context matters for eligibility criteria: the same lab value or medication can mean different things before, during, or after an admission.
- Medication orders and medication administration are different concepts: `prescriptions` records what was ordered, `emar` records what was given.
- Missing evidence should not automatically be interpreted as a negative finding; an absent code or lab may simply reflect incomplete documentation.
- This supports using UNKNOWN as a first-class eligibility state alongside SATISFIED and NOT_SATISFIED.